In [8]:
# Dummy városadatbázis (0-1 normalizált értékek)
cities = {
    "Lisbon": {
        "földrajz": {"tengerpart": 0.9, "hegy": 0.2, "város": 0.7, "sziget": 0.5, "tópart": 0.2, "sivatag": 0.1},
        "ár": 0.9,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.4, "relax": 1.0, "aktív": 0.5, "kulturális": 1.0, "családbarát": 0.6},
        "távolság": 1.0,
        "zsúfoltság": 0.5,
        "egyediség": 0.8
    },
    "Barcelona": {
        "földrajz": {"tengerpart": 1.0, "hegy": 0.1, "város": 0.9, "sziget": 0.3, "tópart": 0.2, "sivatag": 0.0},
        "ár": 0.5,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.6, "relax": 0.5, "aktív": 0.3, "kulturális": 1.0, "családbarát": 0.5},
        "távolság": 0.5,
        "zsúfoltság": 1.0,
        "egyediség": 0.3
    },
    "Tirana": {
        "földrajz": {"tengerpart": 0.4, "hegy": 0.7, "város": 0.6, "sziget": 0.2, "tópart": 0.3, "sivatag": 0.0},
        "ár": 0.9,
        "klíma": 0.8,
        "életstílus": {"bulis": 0.3, "relax": 0.5, "aktív": 0.6, "kulturális": 1.0, "családbarát": 0.4},
        "távolság": 1.0,
        "zsúfoltság": 0.2,
        "egyediség": 0.8
    }
}

# Dummy user input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
        "egyediség": 6
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
    "egyediség": 0.2
}

In [9]:
# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total

In [10]:
# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")

Top ajánlott városok a preferenciáid alapján:
Lisbon: 0.841
Tirana: 0.795
Barcelona: 0.750
